In [5]:
# XGBoost_Aurora.py
# -------------------------------------------------------------
# Tuned XGBoost for aurora intensity prediction
# COMPSCI 760 — Group 10
# -------------------------------------------------------------

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# -------------------------------------------------------------
# 0. Global knobs
# -------------------------------------------------------------
PIPELINE_MODE = True
REFIT_METRIC = "mae"   # or "rmse"

FAST_MODE   = True
CPU_COUNT   = os.cpu_count() or 2
N_JOBS      = min(4, CPU_COUNT)
N_SPLITS    = 2 if FAST_MODE else 3
N_ITER      = 20 if FAST_MODE else 50

# -------------------------------------------------------------
# 1. Load dataset (from Kaggle input)
# -------------------------------------------------------------
BASE_DIR = "/kaggle/input/final-planb-24"
CSV_PATH = os.path.join(BASE_DIR, "final-planb-24.csv")

df = pd.read_csv(CSV_PATH, parse_dates=["time"])
print("Loaded:", CSV_PATH)
print("Shape before drop:", df.shape)
print("Time range:", df["time"].min(), "->", df["time"].max())

# -------------------------------------------------------------
# 2. Define target/features
# -------------------------------------------------------------
TARGET_COL = "keogram_mean"
df = df.dropna(subset=[TARGET_COL]).copy()

drop_cols = ["time", "keogram_mean", "keogram_median", "keogram_max"]
features = [c for c in df.columns if c not in drop_cols]

X_all = df[features]

# -------------------------------------------------------------
# 3. Time-based split
# -------------------------------------------------------------
train_idx = df[(df["time"] < "2018-01-01")].index
val_idx   = df[(df["time"] >= "2018-01-01") & (df["time"] < "2019-01-01")].index
test_idx  = df[(df["time"] >= "2019-01-01") & (df["time"] < "2021-01-01")].index

X_train_df = X_all.loc[train_idx]
X_val_df   = X_all.loc[val_idx]
X_test_df  = X_all.loc[test_idx]

y_train = df.loc[train_idx, TARGET_COL].values
y_val   = df.loc[val_idx, TARGET_COL].values
y_test  = df.loc[test_idx, TARGET_COL].values

if not PIPELINE_MODE:
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train_df)
    X_val   = imputer.transform(X_val_df)
    X_test  = imputer.transform(X_test_df)
else:
    X_train, X_val, X_test = X_train_df, X_val_df, X_test_df

print("Split sizes:",
      "train =", len(train_idx),
      "val =", len(val_idx),
      "test =", len(test_idx))

# -------------------------------------------------------------
# 4. Hyperparameter search
# -------------------------------------------------------------
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
}

param_space = {
    "xgb__n_estimators": [300, 600, 900] if FAST_MODE else [300, 600, 900, 1200],
    "xgb__max_depth": [3, 6, 9] if FAST_MODE else [3, 6, 9, 12],
    "xgb__learning_rate": [0.05, 0.1, 0.2],
    "xgb__subsample": [0.7, 0.9, 1.0],
    "xgb__colsample_bytree": [0.7, 0.9, 1.0],
    "xgb__min_child_weight": [1, 3, 5],
    "xgb__gamma": [0, 0.1, 0.3],
    "xgb__reg_lambda": [1, 5, 10],
}

base_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=N_JOBS,
    tree_method="gpu_hist",    # GPU acceleration
    predictor="gpu_predictor"
)

if PIPELINE_MODE:
    estimator = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("xgb", base_xgb)
    ])
else:
    estimator = base_xgb

search = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=param_space,
    n_iter=N_ITER,
    cv=tscv,
    scoring=scoring,
    refit=REFIT_METRIC,
    n_jobs=N_JOBS,
    verbose=1,
    random_state=42,
)

print("\n[Search] XGBoost ...")
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print(f"Best CV {REFIT_METRIC.upper()}: {-search.best_score_:.4f}")

best_estimator = search.best_estimator_

# -------------------------------------------------------------
# 5. Evaluate
# -------------------------------------------------------------
def eval_and_print(split_name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    print(f"{split_name} -> MSE: {mse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")

val_pred  = best_estimator.predict(X_val_df if PIPELINE_MODE else X_val)
test_pred = best_estimator.predict(X_test_df if PIPELINE_MODE else X_test)

print("\n=== Evaluation ===")
eval_and_print("VAL ", y_val, val_pred)
eval_and_print("TEST", y_test, test_pred)

# -------------------------------------------------------------
# 6. Baselines
# -------------------------------------------------------------
def baseline_report(y_true, name="mean"):
    if name == "mean":
        yhat = np.full_like(y_true, np.mean(y_true), dtype=float)
    elif name == "median":
        yhat = np.full_like(y_true, np.median(y_true), dtype=float)
    mse = mean_squared_error(y_true, yhat)
    mae = mean_absolute_error(y_true, yhat)
    rmse = np.sqrt(mse)
    print(f"Baseline ({name}) -> RMSE: {rmse:.4f}  MAE: {mae:.4f}")

print("\n=== Baselines ===")
baseline_report(y_val, "mean")
baseline_report(y_val, "median")
baseline_report(y_test, "mean")
baseline_report(y_test, "median")

# -------------------------------------------------------------
# 7. Feature importances
# -------------------------------------------------------------
if PIPELINE_MODE:
    model = best_estimator.named_steps["xgb"]
else:
    model = best_estimator

importances = model.feature_importances_
order = np.argsort(importances)[::-1][:15]
print("\nTop-15 features:")
for idx in order:
    print(f"{features[idx]:20s} {importances[idx]:.4f}")


Loaded: /kaggle/input/final-planb-24/final-planb-24.csv
Shape before drop: (78957, 23)
Time range: 2012-01-01 00:00:00 -> 2021-01-01 05:00:00
Split sizes: train = 7430 val = 849 test = 1865

[Search] XGBoost ...
Fitting 2 folds for each of 20 candidates, totalling 40 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [04:36:34] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [04:36:34] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [04:36:34] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserW

Best params: {'xgb__subsample': 0.7, 'xgb__reg_lambda': 5, 'xgb__n_estimators': 300, 'xgb__min_child_weight': 1, 'xgb__max_depth': 3, 'xgb__learning_rate': 0.05, 'xgb__gamma': 0.1, 'xgb__colsample_bytree': 0.9}
Best CV MAE: 22.2808

=== Evaluation ===
VAL  -> MSE: 571.4036  MAE: 17.1047  R2: 0.1605
TEST -> MSE: 502.4839  MAE: 16.6537  R2: 0.0893

=== Baselines ===
Baseline (mean) -> RMSE: 26.0891  MAE: 18.7218
Baseline (median) -> RMSE: 27.8812  MAE: 16.7919
Baseline (mean) -> RMSE: 23.4890  MAE: 17.2677
Baseline (median) -> RMSE: 25.2909  MAE: 15.4169

Top-15 features:
Kp                   0.2718
ap                   0.0929
tp_mm_mean_aw        0.0680
tp_mm_max            0.0642
tp_mm_median         0.0565
temp_p75             0.0392
tp_mm_p75            0.0365
tcc_median           0.0363
temp_max             0.0356
humidity_p75         0.0349
temp_mean            0.0349
tcc_mean             0.0335
temp_median          0.0328
humidity_max         0.0324
tcc_p75              0.0298


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [04:38:44] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [04:38:44] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [3]:
# XGBoost_LogFix.py
# -------------------------------------------------------------
# XGBoost for aurora intensity prediction (with log1p target transform)
# Author: Group 10 (COMPSCI 760) — Kaggle-ready
# -------------------------------------------------------------

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# -------------------------------------------------------------
# 0. Global knobs
# -------------------------------------------------------------
PIPELINE_MODE = True
REFIT_METRIC = "mae"
FAST_MODE = True
N_SPLITS = 2 if FAST_MODE else 3
N_JOBS = 4

# -------------------------------------------------------------
# 1. Load dataset
# -------------------------------------------------------------
BASE_DIR = "/kaggle/input/final-planb-24"
CSV_PATH = os.path.join(BASE_DIR, "final-planb-24.csv")

df = pd.read_csv(CSV_PATH, parse_dates=["time"])
print("Loaded:", CSV_PATH)
print("Shape before drop:", df.shape)
print("Time range:", df["time"].min(), "->", df["time"].max(), flush=True)

# -------------------------------------------------------------
# 2. Target + features
# -------------------------------------------------------------
TARGET_COL = "keogram_mean"

# Drop rows with NaN target
before = len(df)
df = df.dropna(subset=[TARGET_COL]).copy()
after = len(df)
print(f"Dropped rows with NaN target ({TARGET_COL}): {before - after}", flush=True)

# Drop columns not usable as features
drop_cols = ["time", "keogram_mean", "keogram_median", "keogram_max"]
features = [c for c in df.columns if c not in drop_cols]
X_all = df[features]

# -------------------------------------------------------------
# 3. Time-based splits
# -------------------------------------------------------------
train_idx = df[(df["time"] < "2018-01-01")].index
val_idx   = df[(df["time"] >= "2018-01-01") & (df["time"] < "2019-01-01")].index
test_idx  = df[(df["time"] >= "2019-01-01") & (df["time"] < "2021-01-01")].index

print("Split sizes:",
      "train =", len(train_idx),
      "val =", len(val_idx),
      "test =", len(test_idx), flush=True)

X_train_df, X_val_df, X_test_df = (
    X_all.loc[train_idx],
    X_all.loc[val_idx],
    X_all.loc[test_idx],
)

# ✅ Fixed indexing (slice from df, not y_all)
y_train = np.log1p(df.loc[train_idx, TARGET_COL].values)
y_val   = np.log1p(df.loc[val_idx, TARGET_COL].values)
y_test  = np.log1p(df.loc[test_idx, TARGET_COL].values)

# -------------------------------------------------------------
# 4. Model + Hyperparameter search
# -------------------------------------------------------------
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"}

param_dist = {
    "xgb__n_estimators": [300, 600] if FAST_MODE else [300, 600, 900],
    "xgb__max_depth": [3, 5, 7],
    "xgb__learning_rate": [0.05, 0.1],
    "xgb__subsample": [0.7, 0.9, 1.0],
    "xgb__colsample_bytree": [0.7, 0.9, 1.0],
    "xgb__gamma": [0, 0.1, 0.3],
    "xgb__min_child_weight": [1, 3],
    "xgb__reg_lambda": [1, 5, 10],
}

xgb = XGBRegressor(
    objective="reg:squarederror",
    tree_method="gpu_hist",  # fast + GPU-ready
    random_state=42,
    n_jobs=N_JOBS,
)

pipe = Pipeline(steps=[("imp", SimpleImputer(strategy="median")), ("xgb", xgb)])

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=20 if FAST_MODE else 40,
    cv=tscv,
    scoring=scoring,
    refit=REFIT_METRIC,
    verbose=1,
    random_state=42,
    n_jobs=N_JOBS,
)

print("\n[Search] XGBoost ...")
search.fit(X_train_df, y_train)

print("Best params:", search.best_params_)
print(f"Best CV {REFIT_METRIC.upper()}: {-search.best_score_:.4f}")

best_estimator = search.best_estimator_

# -------------------------------------------------------------
# 5. Evaluation
# -------------------------------------------------------------
def eval_and_print(name, y_true, y_pred_log):
    # invert log1p
    y_true_orig = np.expm1(y_true)
    y_pred_orig = np.expm1(y_pred_log)

    mse = mean_squared_error(y_true_orig, y_pred_orig)
    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    r2  = r2_score(y_true_orig, y_pred_orig)
    print(f"{name} -> MSE: {mse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}", flush=True)

print("\n=== Evaluation ===", flush=True)
eval_and_print("VAL ", y_val, best_estimator.predict(X_val_df))
eval_and_print("TEST", y_test, best_estimator.predict(X_test_df))

# -------------------------------------------------------------
# 6. Feature importances
# -------------------------------------------------------------
xgb_model = best_estimator.named_steps["xgb"]
importances = xgb_model.feature_importances_
order = np.argsort(importances)[::-1][:15]
print("\nTop-15 features:")
for i in order:
    print(f"{features[i]:20s}  {importances[i]:.4f}")


Loaded: /kaggle/input/final-planb-24/final-planb-24.csv
Shape before drop: (78957, 23)
Time range: 2012-01-01 00:00:00 -> 2021-01-01 05:00:00
Dropped rows with NaN target (keogram_mean): 68813
Split sizes: train = 7430 val = 849 test = 1865

[Search] XGBoost ...
Fitting 2 folds for each of 20 candidates, totalling 40 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:47:42] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:47:42] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:47:42] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserW

Best params: {'xgb__subsample': 1.0, 'xgb__reg_lambda': 10, 'xgb__n_estimators': 300, 'xgb__min_child_weight': 3, 'xgb__max_depth': 3, 'xgb__learning_rate': 0.05, 'xgb__gamma': 0.3, 'xgb__colsample_bytree': 0.9}
Best CV MAE: 0.4270

=== Evaluation ===
VAL  -> MSE: 545.6938  MAE: 15.5158  R2: 0.1983
TEST -> MSE: 445.3672  MAE: 14.8362  R2: 0.1928

Top-15 features:
Kp                    0.4104
ap                    0.0949
tp_mm_mean_aw         0.0544
tp_mm_median          0.0487
tp_mm_max             0.0429
tp_mm_p75             0.0413
tcc_mean              0.0388
tcc_median            0.0314
humidity_max          0.0304
tcc_p75               0.0261
humidity_p75          0.0259
temp_p75              0.0254
temp_mean             0.0245
temp_median           0.0242
temp_max              0.0234


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:48:42] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:48:42] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


In [4]:
from xgboost import XGBRegressor

# -------------------------------------------------------------
# GPU-optimized XGBRegressor (faster + deeper search)
# -------------------------------------------------------------
xgb = XGBRegressor(
    objective="reg:squarederror",
    tree_method="gpu_hist",   # << use GPU histogram algorithm
    predictor="gpu_predictor",# << prediction on GPU as well
    random_state=42,
    n_jobs=N_JOBS,
)

pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
    ("xgb", xgb),
])

# Expanded search space (balanced for Kaggle GPU)
param_dist = {
    "xgb__n_estimators": [500, 1000, 1500],      # more boosting rounds
    "xgb__learning_rate": [0.01, 0.03, 0.05],    # smaller LR with more trees
    "xgb__max_depth": [4, 6, 8],                 # allow deeper trees
    "xgb__min_child_weight": [1, 3, 5],          # regularization
    "xgb__subsample": [0.7, 0.9, 1.0],           # row sampling
    "xgb__colsample_bytree": [0.7, 0.9, 1.0],    # feature sampling
    "xgb__gamma": [0, 0.1, 0.3],                 # split regularization
    "xgb__reg_lambda": [1, 5, 10],               # L2 regularization
    "xgb__reg_alpha": [0, 0.1, 1],               # L1 regularization
}

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=30,                   # more iterations (GPU can handle this)
    cv=tscv,
    scoring=scoring,
    refit=REFIT_METRIC,
    verbose=2,
    random_state=42,
    n_jobs=N_JOBS,
)

print("\n[Search] GPU-Optimized XGBoost ...")
search.fit(X_train_df, y_train)



[Search] GPU-Optimized XGBoost ...
Fitting 2 folds for each of 30 candidates, totalling 60 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:54:43] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:54:43] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:54:43] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserW

RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=2, test_size=None),
                   estimator=Pipeline(steps=[('imp',
                                              SimpleImputer(strategy='median')),
                                             ('xgb',
                                              XGBRegressor(base_score=None,
                                                           booster=None,
                                                           callbacks=None,
                                                           colsample_bylevel=None,
                                                           colsample_bynode=None,
                                                           colsample_bytree=None,
                                                           device=None,
                                                           early_stopping_rounds=None,
                                                           enable_categorical=Fal...
                                        'xgb__gamma': [0, 0.1, 0.3],
                                        'xgb__learning_rate': [0.01, 0.03,
                                                               0.05],
                                        'xgb__max_depth': [4, 6, 8],
                                        'xgb__min_child_weight': [1, 3, 5],
                                        'xgb__n_estimators': [500, 1000, 1500],
                                        'xgb__reg_alpha': [0, 0.1, 1],
                                        'xgb__reg_lambda': [1, 5, 10],
                                        'xgb__subsample': [0.7, 0.9, 1.0]},
                   random_state=42, refit='mae',
                   scoring={'mae': 'neg_mean_absolute_error',
                            'rmse': 'neg_root_mean_squared_error'},
                   verbose=2)

[CV] END xgb__colsample_bytree=1.0, xgb__gamma=0.1, xgb__learning_rate=0.01, xgb__max_depth=8, xgb__min_child_weight=1, xgb__n_estimators=500, xgb__reg_alpha=0, xgb__reg_lambda=1, xgb__subsample=0.7; total time=  13.6s
[CV] END xgb__colsample_bytree=0.7, xgb__gamma=0.3, xgb__learning_rate=0.03, xgb__max_depth=6, xgb__min_child_weight=1, xgb__n_estimators=1000, xgb__reg_alpha=0.1, xgb__reg_lambda=10, xgb__subsample=1.0; total time=  10.0s
[CV] END xgb__colsample_bytree=0.9, xgb__gamma=0.3, xgb__learning_rate=0.01, xgb__max_depth=6, xgb__min_child_weight=3, xgb__n_estimators=500, xgb__reg_alpha=1, xgb__reg_lambda=10, xgb__subsample=0.9; total time=   9.7s
[CV] END xgb__colsample_bytree=0.7, xgb__gamma=0.3, xgb__learning_rate=0.05, xgb__max_depth=6, xgb__min_child_weight=5, xgb__n_estimators=1000, xgb__reg_alpha=0, xgb__reg_lambda=1, xgb__subsample=0.9; total time=   9.7s
[CV] END xgb__colsample_bytree=1.0, xgb__gamma=0, xgb__learning_rate=0.03, xgb__max_depth=8, xgb__min_child_weight=3, 